# Diabetes Prediction using ML pipeline
Modified Notebook with Threshold optimization to handle false negative and false positives

## Load Libraries and Configuration

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, f1_score
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


## Load and QC data

In [2]:
# Loading data

# Load Data
df_train = pd.read_csv('/kaggle/input/playground-series-s5e12/train.csv')

df_test = pd.read_csv('/kaggle/input/playground-series-s5e12/test.csv')

TARGET = "diagnosed_diabetes"

X = df_train.drop(columns=[TARGET, "id"])
y = df_train[TARGET]

test_ids = df_test["id"]
X_test_final = df_test.drop(columns=["id"])


In [3]:
# Feature Groups and Preprocessors
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
    ]
)


In [4]:
# Model Registry

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),

    "RandomForest": RandomForestClassifier(
        n_estimators=20,
        class_weight="balanced",
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=2000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        gamma=0,
        reg_lambda=1.0,
        reg_alpha=0.0,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",      # fast for large data
        random_state=42,
     ),

    "LightGBM": LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.06,
        max_depth=1,
        num_leaves=71,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
}


In [5]:
# Train and Validation Split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)


In [ ]:
# Train and Calibrate All models

results = []
stored_models = {}

for name, model in models.items():

    base_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    base_pipeline.fit(X_train, y_train)
    prob_uncal = base_pipeline.predict_proba(X_val)[:, 1]

    auc_uncal = roc_auc_score(y_val, prob_uncal)

    results.append([name, "Uncalibrated", auc_uncal])
    stored_models[(name, "Uncalibrated")] = base_pipeline

    for method in ["sigmoid", "isotonic"]:
        calibrated = CalibratedClassifierCV(
            base_pipeline,
            method=method,
            cv=5
        )

        calibrated.fit(X_train, y_train)
        prob_cal = calibrated.predict_proba(X_val)[:, 1]
        auc_cal = roc_auc_score(y_val, prob_cal)

        results.append([name, method.capitalize(), auc_cal])
        stored_models[(name, method.capitalize())] = calibrated

In [7]:
# Model Selection

results_df = pd.DataFrame(
    results, columns=["Model", "Calibration", "ROC_AUC"]
).sort_values("ROC_AUC", ascending=False)

results_df


,Model,Calibration,ROC_AUC
7,XGBoost,Sigmoid,0.726291
8,XGBoost,Isotonic,0.726290
6,XGBoost,Uncalibrated,0.725005
10,LightGBM,Sigmoid,0.707069
11,LightGBM,Isotonic,0.707053
9,LightGBM,Uncalibrated,0.707037
0,LogisticRegression,Uncalibrated,0.694757
1,LogisticRegression,Sigmoid,0.694756
2,LogisticRegression,Isotonic,0.694737
5,RandomForest,Isotonic,0.692299


In [8]:
# Refit Best Model on full data
results_df = pd.DataFrame(
    results, columns=["Model", "Calibration", "ROC_AUC"]
).sort_values("ROC_AUC", ascending=False)

results_df

,Model,Calibration,ROC_AUC
7,XGBoost,Sigmoid,0.726291
8,XGBoost,Isotonic,0.726290
6,XGBoost,Uncalibrated,0.725005
10,LightGBM,Sigmoid,0.707069
11,LightGBM,Isotonic,0.707053
9,LightGBM,Uncalibrated,0.707037
0,LogisticRegression,Uncalibrated,0.694757
1,LogisticRegression,Sigmoid,0.694756
2,LogisticRegression,Isotonic,0.694737
5,RandomForest,Isotonic,0.692299


In [9]:
# Refit Best Model

best_model_name, best_calibration, _ = results_df.iloc[0]

final_model = stored_models[(best_model_name, best_calibration)]
final_model.fit(X, y)

# Validation Probabilities

val_prob = final_model.predict_proba(X_val)[:, 1]


In [10]:
# Decision Threshold Optimization

thresholds = np.linspace(0.01, 0.99, 99)
records = []

for t in thresholds:
    y_pred = (val_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    f1 = f1_score(y_val, y_pred)

    records.append([t, sens, spec, f1])

thr_df = pd.DataFrame(
    records, columns=["Threshold", "Sensitivity", "Specificity", "F1"]
)

# Youden's

fpr, tpr, roc_thr = roc_curve(y_val, val_prob)
youden_thr = roc_thr[np.argmax(tpr - fpr)]


In [11]:
# Cost-Sensitive thresholds

C_FN, C_FP = 5, 1
costs = []

for t in thresholds:
    y_pred = (val_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
    costs.append(C_FN * fn + C_FP * fp)

cost_thr = thresholds[np.argmin(costs)]

In [12]:
# Final Deployment

DEPLOYMENT_THRESHOLD = cost_thr
DEPLOYMENT_THRESHOLD


np.float64(0.25)

In [13]:
# Deploy to Test Set

test_prob = final_model.predict_proba(X_test_final)[:, 1]

submission = pd.DataFrame({
    "id": test_ids,
    "diabetes_probability": test_prob
})

submission.head()


,id,diabetes_probability
0,700000,0.477596
1,700001,0.707971
2,700002,0.783363
3,700003,0.369615
4,700004,0.890520


In [14]:
submission.to_csv("submission.csv", index=False)

print("Deployment file saved: submission.csv")

Deployment file saved: submission.csv
